# Turn SEC EDGAR into a knowledge graph — in one call

SEC filings look like the classic "it's all just tables" domain: companies, filings, insider trades, board members. But the questions investors actually ask are **graph-shaped** — *who sits on two boards? which insiders sold right after an 8-K? does this number trace to a real filing?* — and in SQL those become multi-join queries the planner can't index by relationship.

**`SEC.fetch` builds the graph in one call** — name the forms, the companies, a span:

```python
g = SEC.fetch(path, "13F-HR", "TSLA", years=2, user_agent="Name email@dom")
```

It downloads the filings from SEC EDGAR (with a progress bar), parses them, and returns a Cypher-queryable `KnowledgeGraph`. Re-running the same scope is a cache hit.

Two things make it more than a database:

1. **The same person is one node.** Tim Cook is officer *and* director at Apple — a single `:Person`. Elon Musk is the trifecta at Tesla: officer, director, 10% owner. Every Form 4 trade he files and his board seat hang off *that one node* — no identity reconciliation, no `JOIN ... ON name`.
2. **Every fact is its own typed node.** An insider trade, an 8-K event, a compensation record, a board role — each is a node with its own properties, wired by typed edges (`BY_INSIDER`, `AT_COMPANY`, `REPORTED_IN`). Every fact links back to the filing it was extracted from.

Below: a ~3-minute build (10 companies across sectors, 2 years), seven Cypher queries with no clean SQL equivalent, then the graph handed to Claude Desktop as an MCP server.

Requires `pip install kglite` and — for the last step — the Claude Desktop app. SEC's fair-access policy needs a descriptive User-Agent (your name + a contact email).

In [ ]:
from pathlib import Path

from kglite.datasets.sec import SEC

# SEC's fair-access policy requires a descriptive User-Agent — your
# name + a contact email. Generic UAs get 403'd.
USER_AGENT = "Your Name your@email.com"

# SEC.fetch — the one-call graph builder. Name the forms, the
# companies, and a span; get a KnowledgeGraph back. `forms` and
# `companies` each take a single value or a list; tickers resolve to
# CIKs automatically. A tqdm progress bar tracks the rate-limited
# download (SEC caps at 10 requests/second).
graph = SEC.fetch(
    Path.home() / ".kglite-sec-demo",
    forms=["4", "8-K", "DEF 14A"],  # insider trades, 8-K events, proxies
    companies=[
        "AAPL",
        "MSFT",
        "GOOGL",
        "NVDA",  # big tech
        "BRK-B",
        "JPM",
        "GS",  # finance
        "TSLA",  # auto
        "WMT",
        "HD",  # retail
    ],
    years=2,
    user_agent=USER_AGENT,
)
print(f"\n{graph}\n")

# Cypher: MATCH (var:NodeType)-[:EDGE_TYPE]->(other) ... RETURN ...
# Full reference: https://kglite.readthedocs.io (CYPHER.md)
print("Companies in scope:")
for r in graph.cypher(
    "MATCH (c:Company) RETURN c.title AS company, c.tickers AS tickers, c.sic_description AS sector ORDER BY company"
):
    print(f"  {r['company']:30} {str(r['tickers']):14} {r['sector']}")

## Seven queries the graph wins on

Each one walks typed edges between typed nodes. The SQL equivalents need 3–5 joins and a planner that can't index on relationship type — here they're a single `MATCH` pattern.

In [ ]:
# Q1 — One person, many hats. The unified :Person means an insider
# who is officer AND director AND 10% owner collapses to a single
# node with multiple :Role edges back to the company.
print("Q1 — Multi-role insiders")
for r in graph.cypher("""
    MATCH (c:Company)<-[:AT_COMPANY]-(r:Role)-[:OF_PERSON]->(p:Person)
    WITH p, c, collect(DISTINCT r.role_type) AS roles
    WHERE size(roles) >= 2
    RETURN p.title AS person, c.title AS company, roles
    ORDER BY size(roles) DESC, person LIMIT 10
"""):
    print(f"  {r['person']:26} {r['company']:24} {r['roles']}")

# Q2 — Same node, every source. This joins DEF 14A (board roles) and
# Form 4 (trades) through one :Person — there is no name matching,
# the node itself IS the join.
print("\nQ2 — Board role + insider trading, unified per person")
for r in graph.cypher("""
    MATCH (p:Person)<-[:OF_PERSON]-(r:Role)-[:AT_COMPANY]->(c:Company)
    MATCH (p)<-[:BY_INSIDER]-(t:InsiderTransaction)
    RETURN p.title AS person, c.title AS company,
           collect(DISTINCT r.role_type) AS board_roles,
           count(DISTINCT t) AS form4_trades
    ORDER BY form4_trades DESC LIMIT 10
"""):
    print(f"  {r['person']:24} {r['company']:20} {str(r['board_roles']):34} {r['form4_trades']:>4} trades")

In [ ]:
# Q3 — Insider sell leaderboard. :InsiderTransaction is a node;
# BY_INSIDER and IN_COMPANY are typed edges, so the aggregate is a
# 2-hop MATCH rather than a 3-table join.
print("Q3 — Largest insider sells by dollar value")
for r in graph.cypher("""
    MATCH (t:InsiderTransaction)-[:BY_INSIDER]->(p:Person)
    MATCH (t)-[:IN_COMPANY]->(c:Company)
    WHERE t.direction = 'sale'
    RETURN p.title AS insider, c.title AS company,
           sum(t.shares) AS shares, sum(t.total_value) AS dollars
    ORDER BY dollars DESC LIMIT 10
"""):
    print(f"  {r['insider']:26} {r['company']:20} {(r['shares'] or 0):>13,.0f} sh  ${(r['dollars'] or 0):>16,.0f}")

# Q4 — Pre-rolled insider activity. The loader also materialises an
# :InsiderActivity summary node per (person, company) — trade counts
# and dollar totals already aggregated, no GROUP BY needed.
print("\nQ4 — Insider activity rollup")
for r in graph.cypher("""
    MATCH (a:InsiderActivity)-[:ACTIVITY_OF]->(p:Person)
    MATCH (a)-[:ACTIVITY_AT]->(c:Company)
    RETURN p.title AS insider, c.title AS company,
           a.n_transactions AS trades, a.sell_value AS sell_value
    ORDER BY trades DESC LIMIT 10
"""):
    print(
        f"  {r['insider']:26} {r['company']:20} {(r['trades'] or 0):>4} trades  ${(r['sell_value'] or 0):>16,.0f} sold"
    )

In [ ]:
# Q5 — 8-K material events. Each :CorporateEvent node carries the
# SEC Item code and its official description.
print("Q5 — 8-K events by Item code")
for r in graph.cypher("""
    MATCH (e:CorporateEvent)-[:AT_COMPANY]->(c:Company)
    RETURN c.title AS company, e.item_code AS item, e.title AS event,
           count(*) AS n
    ORDER BY n DESC LIMIT 10
"""):
    print(f"  {r['company']:20} item {str(r['item']):5} x{r['n']:<3} {str(r['event'])[:46]}")

# Q6 — Executive compensation (DEF 14A Item 402). One :Compensation
# node per (executive, fiscal year), linked to person and company.
print("\nQ6 — Executive compensation")
for r in graph.cypher("""
    MATCH (comp:Compensation)-[:OF_PERSON]->(p:Person)
    MATCH (comp)-[:AT_COMPANY]->(c:Company)
    RETURN p.title AS executive, c.title AS company,
           comp.fiscal_year AS fy, comp.total AS total
    ORDER BY total DESC LIMIT 10
"""):
    print(f"  {r['executive']:30} {r['company']:18} FY{r['fy']}  ${(r['total'] or 0):>14,.0f}")

# Q7 — Interlocking directorates. A person on two companies' boards
# is one :Person with :Role edges to both — the cross-company link
# is a single pattern, not a self-join over a names table.
print("\nQ7 — Directors who sit on multiple boards in this graph")
rows = list(
    graph.cypher("""
    MATCH (c1:Company)<-[:AT_COMPANY]-(:Role)-[:OF_PERSON]->(p:Person)<-[:OF_PERSON]-(:Role)-[:AT_COMPANY]->(c2:Company)
    WHERE c1.title < c2.title
    RETURN p.title AS person, c1.title AS company_a, c2.title AS company_b
    LIMIT 10
""")
)
for r in rows:
    print(f"  {r['person']:26} {r['company_a']}  <->  {r['company_b']}")
if not rows:
    print("  (none in this 10-company slice — widen `companies` to surface interlocks)")

In [ ]:
# graph.describe() emits a compact XML schema — every node type and
# its properties, every typed edge and its direction, plus
# exploration hints. It's small enough to drop into a system prompt,
# and it's how the MCP server tells Claude what's in the graph
# before it writes its first cypher_query().
xml = graph.describe()
print(xml[:900])
print(f"\n... [truncated — full schema is {len(xml):,} chars]")

## Hand the same graph to Claude Desktop

The next cell saves the graph and registers it as a Claude Desktop MCP server. After that you can ask Claude things like *"who sold the most stock as an insider this year?"* — it reads the schema with `graph_overview()`, writes Cypher against `:Person → :InsiderTransaction → :Company`, and answers from real filing data.

This uses the `--graph X.kgl` MCP pattern (one curated graph), in contrast to the `workspace` clone-on-demand pattern in [`examples/codebase_to_claude_mcp.ipynb`](https://github.com/kkollsga/kglite/blob/main/examples/codebase_to_claude_mcp.ipynb).

In [ ]:
from pathlib import Path

from kglite.mcp_server import claude_config

DRY_RUN = True  # flip to False to write the Claude Desktop config

kgl_path = Path.home() / ".kglite-sec-demo" / "sec.kgl"
graph.save(str(kgl_path))
print(f"Saved graph: {kgl_path}")

# Tiny manifest. `save_graph: true` lets Claude persist Cypher
# SET/CREATE annotations back to the .kgl file (opt-in since 0.9.41).
manifest_path = kgl_path.parent / "sec_mcp.yaml"
manifest_path.write_text("""\
name: SEC Filings
builtins:
  save_graph: true
instructions: |
  SEC EDGAR knowledge graph — 10 companies, 2 years of Form 4 / 8-K /
  DEF 14A filings. Use cypher_query + graph_overview to answer
  investor questions.
  - :Person unifies Form 4 reporters and DEF 14A directors — one node
    per human, linked to :Company through :Role (AT_COMPANY / OF_PERSON).
  - :InsiderTransaction -[:BY_INSIDER]-> :Person and -[:IN_COMPANY]-> :Company.
  - :CorporateEvent (8-K) and :Compensation link to :Company via AT_COMPANY.
  - Every fact node carries a -[:REPORTED_IN]-> :Filing edge for provenance.
""")

entry = claude_config.add_mcp(
    name="sec-filings",
    command="kglite-mcp-server",
    args=["--graph", str(kgl_path), "--mcp-config", str(manifest_path)],
    client="claude_desktop",
    overwrite=True,
    dry_run=DRY_RUN,
)

cfg_path = claude_config.default_path("claude_desktop")
print(f"{'Would write' if DRY_RUN else 'Wrote'} entry to {cfg_path}:")
print(entry)
if not DRY_RUN:
    print("\nRestart Claude Desktop, then ask:")
    print("  -> Who sold the most stock as an insider in this graph?")

## What you have now

After restarting Claude Desktop, the **sec-filings** MCP server appears in Claude's tools. Useful prompts:

- *"Show me the largest insider sells over the past two years, by dollar value."*
- *"Which people sit on more than one company's board in this graph?"*
- *"For each company, break down 8-K filings by Item code."*
- *"Who are the highest-paid executives, and how does pay compare across companies?"*

Under the hood Claude calls `graph_overview()` (reads the XML schema), then `cypher_query(...)` against the graph this notebook built, and answers in natural language with real rows.

**Scaling up:**

- `SEC.fetch` is the quick path. For the full knob set use `SEC.open` — separate filing-index vs. payload spans, an explicit storage `mode`, and the `include_subsidiaries` / `include_xbrl_metrics` flags that add Exhibit 21 subsidiary trees and XBRL financial metrics.
- Pass more forms (`"SC 13D"` activist stakes, `"144"` planned sales, `"13F-HR"` institutional holdings) or more companies. Drop `companies` entirely to ingest the full ~6,000-filer universe — the build auto-escalates storage mode (`memory` → `mapped` → `disk`) as the predicted graph grows.
- Re-running the same scope is a cache hit: `raw/` is an immutable SEC mirror, `processed/` regenerates only when `raw/` changes, and the built graph loads directly.

**Next steps:**

- Repo: [github.com/kkollsga/kglite](https://github.com/kkollsga/kglite)
- Docs: [kglite.readthedocs.io](https://kglite.readthedocs.io)
- SEC loader guide: [kglite.readthedocs.io/en/latest/guides/sec.html](https://kglite.readthedocs.io/en/latest/guides/sec.html)
- Companion notebook (code-graph pattern): [`examples/codebase_to_claude_mcp.ipynb`](https://github.com/kkollsga/kglite/blob/main/examples/codebase_to_claude_mcp.ipynb)